# PocketVoice — Live-talk Miku 🎤

**Один раз:** Runtime → Change runtime type → **T4 GPU** → Save.

**Каждый день:** ▶ слева от кода ниже. Жди ~10 мин. URL в конце копируй в новую вкладку Chrome.

Если упадёт — снизу будет `[FAIL] step=X reason=...` — кидаешь мне эту строку.


In [ ]:
import os, sys, subprocess, time, threading, re, urllib.request, json, shutil, traceback

# ╔══ helpers ═══════════════════════════════════════════════════════════════╗
class StepFail(Exception): pass

def step(label):
    print(f"\n▸ {label}", flush=True)
    return time.time()

def ok(t0, msg=""):
    print(f"  [OK] {time.time()-t0:.1f}s {msg}", flush=True)

def fail(name, e):
    print(f"\n[FAIL] step={name} reason={type(e).__name__}: {str(e)[:300]}", flush=True)
    print(traceback.format_exc()[:1500], flush=True)
    raise StepFail(name)

def run(cmd, name, timeout=600, retries=2):
    last = None
    for attempt in range(retries + 1):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
            if r.returncode == 0: return r
            last = RuntimeError(f"rc={r.returncode}\n{r.stderr[-800:]}")
        except subprocess.TimeoutExpired as e:
            last = e
        if attempt < retries:
            print(f"  [retry {attempt+1}/{retries}] {' '.join(cmd[:3])}...", flush=True)
            time.sleep(3)
    fail(name, last)

def dl(url, path, min_bytes=1000, retries=3):
    name = os.path.basename(path)
    for attempt in range(retries):
        try:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            tmp = path + ".part"
            urllib.request.urlretrieve(url, tmp)
            size = os.path.getsize(tmp)
            if size < min_bytes:
                os.remove(tmp)
                raise IOError(f"too small: {size} bytes")
            os.replace(tmp, path)
            return size
        except Exception as e:
            print(f"  [retry dl {attempt+1}/{retries}] {name}: {e}", flush=True)
            time.sleep(5)
    raise IOError(f"failed to download {name}")

try:
    # ╔══ 0/6 pre-flight ════════════════════════════════════════════════════╗
    t0 = step("pre-flight checks")
    try:
        import torch
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA НЕ доступна. Runtime → Change runtime type → T4 GPU → Save → перезапусти ▶")
        gpu_name = torch.cuda.get_device_name(0)
        free_gb = shutil.disk_usage('/content').free / 1e9
        if free_gb < 10:
            raise RuntimeError(f"мало диска: {free_gb:.1f}GB, нужно ≥10")
        ok(t0, f"GPU={gpu_name}, disk free={free_gb:.1f}GB")
    except Exception as e:
        fail("preflight", e)

    # ╔══ 1/6 clone w-okada ═════════════════════════════════════════════════╗
    os.chdir('/content/')
    t0 = step("clone w-okada/voice-changer")
    if not os.path.exists('/content/voice-changer'):
        run(['git','clone','-q','https://github.com/w-okada/voice-changer.git'], "clone")
    ok(t0)
    os.chdir('/content/voice-changer/server')

    # ╔══ 2/6 apt + pip deps ════════════════════════════════════════════════╗
    t0 = step("apt: libportaudio2")
    run(['apt-get','-y','install','libportaudio2','-qq'], "apt")
    ok(t0)

    t0 = step("pip: faiss-cpu + pyworld")
    run([sys.executable,'-m','pip','install','-q','faiss-cpu','pyworld','--no-build-isolation'], "pip_base")
    ok(t0)

    t0 = step("pip: fairseq fork (One-sixth)")
    run([sys.executable,'-m','pip','install','-q','fairseq @ git+https://github.com/One-sixth/fairseq.git'], "pip_fairseq", timeout=900)
    ok(t0)

    t0 = step("pip: w-okada requirements.txt")
    run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], "pip_reqs", timeout=900)
    ok(t0)

    # ╔══ 3/6 download Miku checkpoint ═══════════════════════════════════════╗
    t0 = step("download Miku model + index")
    slot_dir = '/content/voice-changer/server/model_dir/RVC/0'
    miku_pth = f'{slot_dir}/miku.pth'
    miku_idx = f'{slot_dir}/miku.index'
    try:
        if not os.path.exists(miku_pth):
            s = dl('https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/miku_default_rvc.pth',
                   miku_pth, min_bytes=10_000_000)
            print(f"  miku.pth = {s//1024//1024} MB", flush=True)
        if not os.path.exists(miku_idx):
            s = dl('https://huggingface.co/NoCrypt/miku_RVC/resolve/main/1a_miku_default_rvc_(aple)/added_IVF4457_Flat_nprobe_1_miku_default_rvc_v2.index',
                   miku_idx, min_bytes=1_000_000)
            print(f"  miku.index = {s//1024//1024} MB", flush=True)
        # params.json для авторегистрации Miku в Slot 0
        params = {'slotIndex':0,'voiceChangerType':'RVC','name':'Miku','description':'Hatsune Miku (RVC v2)',
                  'modelFile':'miku.pth','indexFile':'miku.index','defaultTune':12,'defaultIndexRatio':0.75,
                  'defaultProtect':0.33,'sampleRate':40000,'modelType':'pyTorchRVCv2','embChannels':768,
                  'embOutputLayer':12,'useFinalProj':False,'f0':True}
        with open(f'{slot_dir}/params.json','w') as f: json.dump(params, f)
    except Exception as e:
        fail("download_miku", e)
    ok(t0)

    # ╔══ 4/6 cloudflared ═══════════════════════════════════════════════════╗
    t0 = step("cloudflared binary")
    try:
        if not os.path.exists('/content/cloudflared'):
            dl('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
               '/content/cloudflared', min_bytes=10_000_000)
            os.chmod('/content/cloudflared', 0o755)
    except Exception as e:
        fail("cloudflared_dl", e)
    ok(t0)

    # ╔══ 5/6 start server ══════════════════════════════════════════════════╗
    t0 = step("start w-okada server :18888")
    try:
        server = subprocess.Popen(
            [sys.executable,'MMVCServerSIO.py','-p','18888','--https','False',
             '--content_vec_500','pretrain/checkpoint_best_legacy_500.pt',
             '--hubert_base','hubert_base.pt','--rmvpe','rmvpe.pt','--colab','True'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        # Ждём готовности до 5 мин (HuBERT/RMVPE подкачаются автоматом)
        ready = False
        deadline = time.time() + 300
        for line in iter(server.stdout.readline, ''):
            print("  srv:", line.rstrip(), flush=True)
            if 'Uvicorn running' in line or 'Application startup complete' in line or 'http://0.0.0.0:18888' in line:
                ready = True
                break
            if time.time() > deadline:
                raise TimeoutError("server не стартанул за 5 мин")
            if server.poll() is not None:
                raise RuntimeError(f"server вышел rc={server.returncode}")
        if not ready: raise RuntimeError("server не дал ready-сигнала")
        time.sleep(3)
    except Exception as e:
        fail("start_server", e)
    ok(t0)

    # ╔══ 6/6 cloudflared tunnel ═════════════════════════════════════════════╗
    t0 = step("cloudflared tunnel")
    public_url = None
    try:
        tunnel = subprocess.Popen(
            ['/content/cloudflared','tunnel','--url','http://localhost:18888','--no-autoupdate'],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        url_re = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
        deadline = time.time() + 90
        for line in iter(tunnel.stdout.readline, ''):
            print("  tun:", line.rstrip(), flush=True)
            m = url_re.search(line)
            if m: public_url = m.group(0); break
            if time.time() > deadline:
                raise TimeoutError("cloudflared не дал URL за 90 сек")
            if tunnel.poll() is not None:
                raise RuntimeError(f"cloudflared вышел rc={tunnel.returncode}")
        if not public_url: raise RuntimeError("URL не извлекся")
    except Exception as e:
        fail("tunnel", e)
    ok(t0, public_url)

    # ╔══ READY ═════════════════════════════════════════════════════════════╗
    print("\n" + "═"*72, flush=True)
    print(f"║  🎤 LIVE-TALK MIKU URL:", flush=True)
    print(f"║  {public_url}", flush=True)
    print("═"*72, flush=True)
    print("\n👉 Долгий тап на URL → Copy → новая вкладка Chrome → вставить", flush=True)
    print("👉 В w-okada: Slot 0 = Miku → Start → mic → говори\n", flush=True)
    print("⚠ НЕ ЗАКРЫВАЙ эту вкладку Colab. Idle 6h → отрубится.\n", flush=True)

    def tail(p, tag):
        for ln in iter(p.stdout.readline,''): print(f"  [{tag}] {ln}", end='', flush=True)
    threading.Thread(target=tail, args=(server,'srv'), daemon=True).start()
    threading.Thread(target=tail, args=(tunnel,'tun'), daemon=True).start()

    while True:
        time.sleep(60)
        if server.poll() is not None:
            print(f"\n[FAIL] step=runtime reason=server died rc={server.returncode}")
            break
        if tunnel.poll() is not None:
            print(f"\n[FAIL] step=runtime reason=tunnel died rc={tunnel.returncode}")
            break

except StepFail:
    pass
except Exception as e:
    fail("unknown", e)
